# 01. 데이터 수집 · 정리
- AI Hub train/val JSON 경로 수집
- JSON 파싱 → DataFrame 생성
- 필터링 → 중복 제거 → 층화 샘플링
- 최종 dataset.csv 저장
- 선정 이미지 격리 복사 (raw_sub)

In [5]:
import os
import json
import shutil   # 파일/폴더를 복사·이동·삭제하는 도구
import hashlib  # 파일의 고유 지문(해시값)을 계산하는 도구
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor   # 여러 작업을 동시에 병렬로 처리하는 도구
from sklearn.model_selection import train_test_split
from tqdm import tqdm   # 진행 상황을 progress bar로 보여주는 도구

print('라이브러리 로드 완료')

라이브러리 로드 완료


## 1. 경로 설정 및 이미지 인덱싱

In [6]:
# ── 경로 설정 ──
# 노트북이 notebooks/ 안에 있으므로 ROOT는 한 단계 위
ROOT         = Path('..')
AIHUB_TRAIN  = ROOT / 'data/raw/Training'
AIHUB_VAL    = ROOT / 'data/raw/Validation'
IMG_DIR      = ROOT / 'data/raw'
PROCESSED    = ROOT / 'data/processed'
RAW_SUB_DIR  = ROOT / 'data/raw_sub'      # 최종 선정 이미지 격리 폴더

PROCESSED.mkdir(parents=True, exist_ok=True)
RAW_SUB_DIR.mkdir(parents=True, exist_ok=True)

# 경로 확인
print('ROOT       :', ROOT.resolve())
print('Training   :', AIHUB_TRAIN.exists())
print('Validation :', AIHUB_VAL.exists())

# ── 이미지 경로 인덱싱 (한 번만 실행 → 이후 탐색 즉시 가능) ──
print('\n이미지 경로 인덱싱 중...')
img_index = {
    p.name: p
    for p in tqdm(IMG_DIR.rglob('*.jpg'), desc='jpg 탐색')
}
print(f'인덱싱 완료: {len(img_index):,}개')

ROOT       : D:\project\SAG
Training   : True
Validation : True

이미지 경로 인덱싱 중...


jpg 탐색: 496068it [00:22, 21611.53it/s]

인덱싱 완료: 496,068개


## 2. AI Hub train/val JSON 경로 수집

In [7]:
# AI Hub train/val JSON 경로 수집 (복사 없이 바로 사용)
json_paths = (
    list(tqdm(AIHUB_TRAIN.rglob('*.json'), desc='Training 탐색')) +
    list(tqdm(AIHUB_VAL.rglob('*.json'),   desc='Validation 탐색'))
)
print(f'\n총 JSON 수: {len(json_paths):,}개')

Training 탐색: 440945it [00:03, 111077.81it/s]
Validation 탐색: 55117it [00:00, 70549.01it/s]


총 JSON 수: 496,062개


## 3. JSON 파싱 → DataFrame

In [8]:
def parse_json(json_path):
    """JSON 1개 파싱 → dict 반환. 실패 시 None."""
    try:
        with open(json_path, encoding='utf-8') as f:
            j = json.load(f)
        meta   = j.get('metaData', {})
        lesion = meta.get('lesions', '')

        # 무증상(A7) 예외처리: lesions 키가 비어있을 때 Path 필드로 폴백
        if not lesion:
            path_type = meta.get('Path', '')
            if '무증상' in path_type:
                lesion = 'A7'

        # 유효한 클래스(A1~A7)가 아니면 스킵
        if not lesion or not str(lesion).startswith('A'):
            return None

        return {
            'json_path': str(json_path),
            'img_file' : meta.get('Raw data ID', ''),
            'species'  : meta.get('species', ''),    # D=개, C=고양이
            'lesion'   : lesion,                     # A1~A7
            'region'   : meta.get('region', ''),     # B/L/H/A
            'path_type': meta.get('Path', ''),       # 유증상/무증상
        }
    except Exception:
        return None

with ThreadPoolExecutor(max_workers=8) as exe:
    records = list(tqdm(exe.map(parse_json, json_paths), total=len(json_paths), desc='JSON 파싱'))

df = pd.DataFrame([r for r in records if r is not None])
print(f'\n파싱 완료: {len(df):,}개 (실패/스킵: {len(json_paths) - len(df):,}개)')
df.head()

JSON 파싱: 100%|██████████| 496062/496062 [28:46<00:00, 287.30it/s]  



파싱 완료: 488,103개 (실패/스킵: 7,959개)


,json_path,img_file,species,lesion,region,path_type
0,..\data\raw\Training\TL01\반려견\피부\일반카메라\무증상\A1_...,IMG_D_A7_207463.jpg,D,A7,L,무증상
1,..\data\raw\Training\TL01\반려견\피부\일반카메라\무증상\A1_...,IMG_D_A7_207464.jpg,D,A7,B,무증상
2,..\data\raw\Training\TL01\반려견\피부\일반카메라\무증상\A1_...,IMG_D_A7_207465.jpg,D,A7,L,무증상
3,..\data\raw\Training\TL01\반려견\피부\일반카메라\무증상\A1_...,IMG_D_A7_207466.jpg,D,A7,H,무증상
4,..\data\raw\Training\TL01\반려견\피부\일반카메라\무증상\A1_...,IMG_D_A7_207467.jpg,D,A7,B,무증상


## 4. 필터링

In [9]:
# 축종 확장(D=반려견, C=반려묘) + 유효 피부 질환 클래스(A1~A7) + 로컬에 실제 이미지가 존재하는 데이터만 필터링
df_filtered = df[
    (df['species'].isin(['D', 'C'])) &      # 개(Dog)와 고양이(Cat) 모두 포함
    (df['lesion'].str.match(r'^A[1-7]$')) & # 구진부터 무증상까지 타겟팅
    (df['img_file'].isin(img_index))        # 파일 매핑이 입증된 표본만 선별
].copy()

print(f'필터링 전: {len(df):,}장')
print(f'필터링 후: {len(df_filtered):,}장')
print('\n클래스별 분포:')
print(df_filtered['lesion'].value_counts().sort_index())

필터링 전: 488,103장
필터링 후: 481,193장

클래스별 분포:
lesion
A1     36607
A2     78249
A3     60708
A4     20978
A5     10883
A6     21395
A7    252373
Name: count, dtype: int64


## 5. 중복 이미지 제거 (MD5 해시)

In [10]:
def get_hash(img_file):
    """이미지 파일 MD5 해시 반환. 실패 시 None."""
    try:
        path = img_index.get(img_file)
        if path is None:
            return None
        return hashlib.md5(open(path, 'rb').read()).hexdigest()
    except Exception:
        return None

with ThreadPoolExecutor(max_workers=8) as exe:
    hashes = list(tqdm(exe.map(get_hash, df_filtered['img_file']), total=len(df_filtered), desc='해시 계산'))

df_filtered = df_filtered.copy()
df_filtered['hash'] = hashes

before = len(df_filtered)
df_filtered = df_filtered.dropna(subset=['hash'])         # 파일 없는 것 제거
df_filtered = df_filtered.drop_duplicates(subset='hash')  # 중복 제거
after = len(df_filtered)

print(f'중복 제거 전: {before:,}장')
print(f'중복 제거 후: {after:,}장  (제거: {before - after:,}장)')

해시 계산: 100%|██████████| 481193/481193 [51:46<00:00, 154.87it/s]  


중복 제거 전: 481,193장
중복 제거 후: 432,884장  (제거: 48,309장)


## 6. 클래스 × 부위 층화 샘플링

In [11]:
TARGET_PER_CLASS = 5000
n_regions        = df_filtered['region'].nunique()
per_region       = TARGET_PER_CLASS // n_regions

print(f'촬영 부위 종류: {n_regions}개 → 부위별 목표: {per_region}장')

# groupby apply 대신 직접 샘플링
samples = []
for (lesion, region), group in df_filtered.groupby(['lesion', 'region']):
    samples.append(group.sample(n=min(len(group), per_region), random_state=42))

df_sampled = pd.concat(samples).reset_index(drop=True)

print('\n샘플링 후 클래스별 수량:')
print(df_sampled['lesion'].value_counts().sort_index())
print('\n클래스 × 부위 분포:')
print(df_sampled.groupby(['lesion', 'region']).size().unstack(fill_value=0))

촬영 부위 종류: 4개 → 부위별 목표: 1250장

샘플링 후 클래스별 수량:
lesion
A1    5000
A2    5000
A3    5000
A4    5000
A5    5000
A6    5000
A7    5000
Name: count, dtype: int64

클래스 × 부위 분포:
region     A     B     H     L
lesion                        
A1      1250  1250  1250  1250
A2      1250  1250  1250  1250
A3      1250  1250  1250  1250
A4      1250  1250  1250  1250
A5      1250  1250  1250  1250
A6      1250  1250  1250  1250
A7      1250  1250  1250  1250


In [12]:
# A7 내부 구성 확인
print(df_filtered[df_filtered['lesion'] == 'A7']['path_type'].value_counts())
print(df_filtered[df_filtered['lesion'] == 'A7']['region'].value_counts())

path_type
무증상    223149
Name: count, dtype: int64
region
B    69916
L    64863
H    50871
A    37499
Name: count, dtype: int64


A7 무증상 클래스는 전체가 무증상으로만 구성되어 있음을 확인했고, 촬영 부위 편차가 있었으나 부위별 균등 샘플링으로 편향을 방지했습니다.

## 7. Train / Val / Test 분할 및 CSV 저장

In [13]:
# 70 / 15 / 15 비율로 분할 (stratify로 클래스 균형 유지)
train, temp = train_test_split(
    df_sampled, test_size=0.3,
    stratify=df_sampled['lesion'], random_state=42
)
val, test = train_test_split(
    temp, test_size=0.5,
    stratify=temp['lesion'], random_state=42
)

train = train.copy(); train['split'] = 'train'
val   = val.copy();   val['split']   = 'val'
test  = test.copy();  test['split']  = 'test'

final_df = pd.concat([train, val, test]).reset_index(drop=True)

# img_path(절대경로)를 컬럼으로 저장 → 02_eda에서 바로 사용 가능
final_df['img_path'] = final_df['img_file'].map(lambda x: str(img_index.get(x, '')))
final_df.to_csv(PROCESSED / 'dataset.csv', index=False)

print(f'dataset.csv 저장 완료')
print(f'  train : {len(train):,}장')
print(f'  val   : {len(val):,}장')
print(f'  test  : {len(test):,}장')
print(f'  합계  : {len(final_df):,}장')

dataset.csv 저장 완료
  train : 24,500장
  val   : 5,250장
  test  : 5,250장
  합계  : 35,000장


## 8. 선정 이미지 격리 복사 (raw_sub)
3일차 전처리 속도 향상을 위해 최종 선정된 이미지만 별도 폴더로 복사

In [14]:
# def copy_target_img(row):
#     """선정된 이미지를 raw_sub 폴더로 복사. 이미 존재하면 스킵."""
#     src = row['img_path']
#     if src and os.path.exists(src):
#         dest = RAW_SUB_DIR / os.path.basename(src)
#         if not dest.exists():
#             shutil.copy(src, dest)
#         return str(dest)
#     return ''

# print('선정 이미지 격리 폴더로 복사 중...')
# with ThreadPoolExecutor(max_workers=8) as exe:
#     sub_paths = list(tqdm(
#         exe.map(copy_target_img, [row for _, row in final_df.iterrows()]),
#         total=len(final_df), desc='이미지 복사'
#     ))

# # 격리된 경로를 sub_img_path 컬럼으로 추가 후 CSV 업데이트
# final_df['sub_img_path'] = sub_paths
# final_df.to_csv(PROCESSED / 'dataset.csv', index=False)

# success = sum(1 for p in sub_paths if p)
# print(f'\n복사 완료: {success:,}장 → data/raw_sub/')
# print('dataset.csv 업데이트 완료 (sub_img_path 컬럼 추가)')

## 8. 선정 이미지 병변 부위 크롭(Crop) 및 격리 저장
- 3일 차 모델 학습 속도 및 정확도 향상을 위한 전처리 파이프라인 구축
- AI Hub 라벨링 데이터(JSON)의 Bounding Box 좌표(`x, y, w, h`) 파싱
- 원본 이미지에서 주변 배경 및 털 노이즈를 제거하고, 순수 피부 병변 부위만 정확하게 크롭(Crop)하여 `data/raw_sub/` 폴더에 격리 저장
- 최종 학습용 매니페스트 파일(`dataset.csv`)에 크롭된 이미지 경로(`sub_img_path`) 업데이트

In [15]:
def copy_target_img(row):
    """JSON의 bounding box 좌표를 기반으로 병변 부위만 Crop하여 raw_sub 폴더에 저장"""
    src = row['img_path']
    json_p = row['json_path']  # 3번 셀에서 파싱할 때 담아둔 JSON 절대경로 활용
    
    if src and os.path.exists(src):
        dest = RAW_SUB_DIR / os.path.basename(src)
        if not dest.exists():
            try:
                # 1. JSON 파일 열고 좌표 데이터 파싱
                with open(json_p, 'r', encoding='utf-8') as f:
                    json_data = json.load(f)
                
                # 3번 셀에서 확인한 구조(metaData)를 기반으로 크롭 좌표 추출
                # AI Hub 데이터셋 표준 구조에 따라 metaData -> lesions(또는 메타 내부 좌표)를 파싱합니다.
                meta = json_data.get('metaData', {})
                
                # [중요] AI Hub 세부 버전에 따라 x, y, w, h의 위치가 다를 수 있습니다.
                # 만약 metaData 바로 아래에 x, y, w, h가 있다면 아래대로 작동합니다.
                x = int(meta.get('x', 0))
                y = int(meta.get('y', 0))
                w = int(meta.get('w', 0))
                h = int(meta.get('h', 0))
                
                # 만약 모든 값이 0으로 나오거나 좌표를 못 가져오면 안전장치로 원본 복사
                if w == 0 or h == 0:
                    shutil.copy(src, dest)
                    return str(dest)
                
                # 2. 원본 이미지 로드 후 크롭 (PIL 좌표계: 좌, 상, 우, 하)
                img = Image.open(src)
                cropped_img = img.crop((x, y, x + w, y + h))
                
                # 3. 크롭된 이미지를 지정된 경로에 저장
                cropped_img.save(dest)
                
            except Exception as e:
                # 에러 발생 시 진행에 방해되지 않게 경고만 띄우고 안전하게 원본 복사 처리
                print(f"Crop 실패 ({os.path.basename(src)}): {e}")
                shutil.copy(src, dest) 
                
        return str(dest)
    return ''

print('선정 이미지 병변 부위 크롭 및 격리 폴더 저장 중...')
with ThreadPoolExecutor(max_workers=8) as exe:
    sub_paths = list(tqdm(
        exe.map(copy_target_img, [row for _, row in final_df.iterrows()]),
        total=len(final_df), desc='이미지 크롭 및 복사'
    ))

# 격리된 크롭 경로를 sub_img_path 컬럼으로 추가 후 CSV 업데이트
final_df['sub_img_path'] = sub_paths
final_df.to_csv(PROCESSED / 'dataset.csv', index=False)

success = sum(1 for p in sub_paths if p)
print(f'\n크롭 및 저장 완료: {success:,}장 → data/raw_sub/')
print('dataset.csv 업데이트 완료 (sub_img_path 컬럼 추가)')

선정 이미지 병변 부위 크롭 및 격리 폴더 저장 중...


이미지 크롭 및 복사: 100%|██████████| 35000/35000 [04:14<00:00, 137.67it/s]



크롭 및 저장 완료: 35,000장 → data/raw_sub/
dataset.csv 업데이트 완료 (sub_img_path 컬럼 추가)
